In [87]:
import pandas as pd
df_product = pd.read_csv('Products.csv')
df_product

,ProductKey,ProductName,Brand,Color,UnitCostUSD,UnitPriceUSD,SubcategoryKey,Subcategory,CategoryKey,Category
0,1,Contoso 512MB MP3 Player E51 Silver,Contoso,Silver,6.62,12.99,101,MP4&MP3,1,Audio
1,2,Contoso 512MB MP3 Player E51 Blue,Contoso,Blue,6.62,12.99,101,MP4&MP3,1,Audio
2,3,Contoso 1G MP3 Player E100 White,Contoso,White,7.40,14.52,101,MP4&MP3,1,Audio
3,4,Contoso 2G MP3 Player E200 Silver,Contoso,Silver,11.00,21.57,101,MP4&MP3,1,Audio
4,5,Contoso 2G MP3 Player E200 Red,Contoso,Red,11.00,21.57,101,MP4&MP3,1,Audio
...,...,...,...,...,...,...,...,...,...,...
2512,2513,Contoso Bluetooth Active Headphones L15 Red,Contoso,Red,43.07,129.99,505,Cell phones Accessories,5,Cell phones
2513,2514,Contoso Bluetooth Active Headphones L15 White,Contoso,White,43.07,129.99,505,Cell phones Accessories,5,Cell phones
2514,2515,Contoso In-Line Coupler E180 White,Contoso,White,1.71,3.35,505,Cell phones Accessories,5,Cell phones
2515,2516,Contoso In-Line Coupler E180 Black,Contoso,Black,1.71,3.35,505,Cell phones Accessories,5,Cell phones


In [88]:
df_sale = pd.read_csv('Sales.csv')
df_sale

,OrderNumber,LineItem,OrderDate,DeliveryDate,CustomerKey,StoreKey,ProductKey,Quantity,CurrencyCode
0,366000,1,1/1/2016,NaN,265598,10,1304,1,CAD
1,366001,1,1/1/2016,1/13/2016,1269051,0,1048,2,USD
2,366001,2,1/1/2016,1/13/2016,1269051,0,2007,1,USD
3,366002,1,1/1/2016,1/12/2016,266019,0,1106,7,CAD
4,366002,2,1/1/2016,1/12/2016,266019,0,373,1,CAD
...,...,...,...,...,...,...,...,...,...
62879,2243030,1,2/20/2021,NaN,1216913,43,632,3,USD
62880,2243031,1,2/20/2021,2/24/2021,511229,0,98,4,EUR
62881,2243032,1,2/20/2021,2/23/2021,331277,0,1613,2,CAD
62882,2243032,2,2/20/2021,2/23/2021,331277,0,1717,2,CAD


In [89]:
df_content_base = pd.read_csv('df_content_base.csv')

In [90]:
df_sale_have_productKey_order = df_sale[df_sale['ProductKey'].isin(df_content_base['ProductKey'])]['OrderNumber']
df_sale_have_productKey = df_sale[df_sale['OrderNumber'].isin(df_sale_have_productKey_order)]
#df_sale_have_productKey
df_sale_have_productKey

,OrderNumber,LineItem,OrderDate,DeliveryDate,CustomerKey,StoreKey,ProductKey,Quantity,CurrencyCode
217,372016,1,1/7/2016,NaN,1170323,42,1581,2,GBP
218,372016,2,1/7/2016,NaN,1170323,42,6,3,GBP
271,374016,1,1/9/2016,NaN,1535357,45,9,5,USD
439,385009,1,1/20/2016,NaN,1720050,61,6,1,USD
440,385009,3,1/20/2016,NaN,1720050,61,123,5,USD
...,...,...,...,...,...,...,...,...,...
62734,2241013,7,2/18/2021,2/22/2021,1259633,0,1680,1,USD
62744,2241016,1,2/18/2021,NaN,1743547,66,7,1,USD
62806,2243000,1,2/20/2021,NaN,723572,30,9,8,EUR
62807,2243000,3,2/20/2021,NaN,723572,30,553,9,EUR


In [91]:
df_product

,ProductKey,ProductName,Brand,Color,UnitCostUSD,UnitPriceUSD,SubcategoryKey,Subcategory,CategoryKey,Category
0,1,Contoso 512MB MP3 Player E51 Silver,Contoso,Silver,6.62,12.99,101,MP4&MP3,1,Audio
1,2,Contoso 512MB MP3 Player E51 Blue,Contoso,Blue,6.62,12.99,101,MP4&MP3,1,Audio
2,3,Contoso 1G MP3 Player E100 White,Contoso,White,7.40,14.52,101,MP4&MP3,1,Audio
3,4,Contoso 2G MP3 Player E200 Silver,Contoso,Silver,11.00,21.57,101,MP4&MP3,1,Audio
4,5,Contoso 2G MP3 Player E200 Red,Contoso,Red,11.00,21.57,101,MP4&MP3,1,Audio
...,...,...,...,...,...,...,...,...,...,...
2512,2513,Contoso Bluetooth Active Headphones L15 Red,Contoso,Red,43.07,129.99,505,Cell phones Accessories,5,Cell phones
2513,2514,Contoso Bluetooth Active Headphones L15 White,Contoso,White,43.07,129.99,505,Cell phones Accessories,5,Cell phones
2514,2515,Contoso In-Line Coupler E180 White,Contoso,White,1.71,3.35,505,Cell phones Accessories,5,Cell phones
2515,2516,Contoso In-Line Coupler E180 Black,Contoso,Black,1.71,3.35,505,Cell phones Accessories,5,Cell phones


In [92]:
# Hàm tính lợi nhuận cho mỗi sản phẩm
def calculate_profit(product_key, quantity):
    product = df_product[df_product['ProductKey'] == product_key]
    if not product.empty:
        unit_profit = product['UnitPriceUSD'].values[0] - product['UnitCostUSD'].values[0]
        return round(unit_profit * quantity)
    else:
        return 0

# Tạo ra dữ liệu DB_Utility (ProductKey : Tổng lợi nhuận : Lợi nhuận của từng ProductKey)
db_utility_data = []

for order_num, group in df_sale_have_productKey.groupby('OrderNumber'):
    product_keys = group['ProductKey'].tolist()
    quantities = group['Quantity'].tolist()

    # Chỉ tiếp tục nếu có từ 2 ProductKey trở lên
    if len(product_keys) >= 2:
        # Tính lợi nhuận cho từng ProductKey
        profits = [calculate_profit(product_key, quantity) for product_key, quantity in zip(product_keys, quantities)]
        
        # Tính tổng lợi nhuận cho cả giao dịch
        total_profit = sum(profits)
        
        # Tạo chuỗi DB_Utility theo định dạng yêu cầu
        utility_entry = f"{' '.join(map(str, product_keys))}:{total_profit}:{' '.join(map(str, profits))}"
        db_utility_data.append(utility_entry)

# Kết quả DB_Utility
db_utility_data


['1581 6:325:293 32',
 '6 123:767:11 756',
 '2355 1209 2:2334:851 1426 57',
 '1599 32 2 2006:702:31 512 19 140',
 '37 535:297:200 97',
 '1223 3:683:676 7',
 '1216 1694 1209 6:3791:2344 10 1426 11',
 '853 3:168:154 14',
 '95 34 56:1022:99 329 594',
 '406 1986 1647 1604 4 171 1590:1304:563 137 97 348 21 107 31',
 '1211 1758 7:1021:918 50 53',
 '2458 150 4:2419:10 2377 32',
 '17 3:102:59 43',
 '14 1660:362:168 194',
 '175 187 2002 1592 6 103 1227:1615:63 174 108 39 11 559 661',
 '4 443 38 143:1700:11 754 200 735',
 '1792 2075 1461 1309 1663 8:1694:105 292 1165 96 7 29',
 '101 34:436:389 47',
 '4 1590 1973:526:53 61 412',
 '1261 1388 3:247:54 143 50',
 '964 423 49 3:676:194 324 108 50',
 '2065 508 140 8:965:98 151 540 176',
 '8 1630 1350:66:29 31 6',
 '947 1629 2:105:89 10 6',
 '62 8 1581 1513:1715:196 59 879 581',
 '1738 3:83:69 14',
 '4 418:153:21 132',
 '2210 8:205:176 29',
 '37 1586:414:400 14',
 '37 2094:979:600 379',
 '1572 1158 2:3399:219 3129 51',
 '1748 417 423 8 57:1550:73 971 32

In [93]:
#save db_utility_data to file
with open('db_utility_data.txt', 'w') as f:
    for item in db_utility_data:
        f.write("%s\n" % item)